# Customer Intelligence: Country Segmentation & Predictive Modeling

Developed an end-to-end Customer Intelligence System using clustering (K-Means, DBSCAN) and ensemble learning (Random Forest, XGBoost), achieving optimized predictive performance and actionable insights for country development categories.

## 1) Install required libraries

In [1]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn xgboost

## 2) Import libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

import warnings
warnings.filterwarnings('ignore')
sns.set(style='whitegrid', palette='muted')

## 3) Load the dataset

In [3]:
df = pd.read_csv('Country-data.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Country-data.csv'

## 4) Quick inspection

In [ ]:
print(f"Shape: {df.shape}")
display(df.info())
display(df.describe().T)

## 5) Expected columns

In [ ]:
expected_cols = ['country', 'child_mort', 'exports', 'health', 'imports', 'income', 'inflation', 'life_expec', 'total_fer', 'gdpp']
print("Available columns:", df.columns.tolist())
print("Match expected:", all(col in df.columns for col in expected_cols))

## 6) Basic cleaning

In [ ]:
# Standardizing column names, removing duplicates, and validating expected features
df.columns = df.columns.str.strip().str.lower()
df = df.drop_duplicates().reset_index(drop=True)

expected_cols = ['country', 'child_mort', 'exports', 'health', 'imports', 'income', 'inflation', 'life_expec', 'total_fer', 'gdpp']
missing_cols = [col for col in expected_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

# Convert expected numeric features to numeric and preserve the original country identifier
numeric_cols = expected_cols[1:]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

print(f"Shape after deduplication: {df.shape}")
print("Missing values by expected feature:")
display(df[expected_cols].isnull().sum())
print("Numeric columns identified:", numeric_cols)
display(df.head())

## 7) Exploratory Data Analysis

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

# Outlier detection using Boxplots
plt.figure(figsize=(15, 10))
for i, col in enumerate(numeric_cols):
    plt.subplot(3, 3, i+1)
    sns.boxplot(y=df[col])
    plt.title(f'Outliers in {col}')
plt.tight_layout()
plt.show()

## 8) Feature scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numeric_cols])
print("Feature scaling complete. Data is now normalized for distance-based algorithms.")

## 9) K-Means: Elbow method

In [ ]:
inertias = []
k_range = range(2, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal K')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.show()

## 10) Train K-Means

In [ ]:
# Choosing k=3 based on Elbow and Silhouette analysis
best_k = 3
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

print(f"K-Means training complete with k={best_k}.")
print('Silhouette Score:', silhouette_score(X_scaled, df['kmeans_cluster']))

## 11) Try DBSCAN

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)
print("DBSCAN Cluster Distribution:\n", df['dbscan_cluster'].value_counts())

## 12) PCA visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df['kmeans_cluster'], palette='viridis', s=100)
plt.title('Clusters Visualized through PCA (2 Components)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.show()

## 13) Cluster profiling

In [ ]:
profile = df.groupby('kmeans_cluster')[numeric_cols].mean().round(2)
display(profile)

# Visualizing cluster differences for key indicators
df.groupby('kmeans_cluster')[['gdpp', 'income', 'child_mort']].mean().plot(kind='bar', figsize=(12, 6))
plt.title('Mean Socio-Economic Indicators by Cluster')
plt.show()

## 14) Final insights & Predictive Performance

In [ ]:
print("### Actionable Insights")
print("- Cluster 1: High-priority countries with high child mortality and low income. Immediate aid target.")
print("- Cluster 2: Developing nations with moderate economic indicators and growth potential.")
print("- Cluster 0: Developed nations with high GDP per capita and healthy social indicators.")

print("\n### Predictive Performance (Ensemble Learning)")
X = df[numeric_cols]
y = df['kmeans_cluster']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print("Random Forest Classification Report:\n", classification_report(y_test, rf.predict(X_test)))

# XGBoost
if XGB_AVAILABLE:
    xgb = XGBClassifier(eval_metric='mlogloss', random_state=42)
    xgb.fit(X_train, y_train)
    print("XGBoost Classification Report:\n", classification_report(y_test, xgb.predict(X_test)))

# Feature Importance
pd.Series(rf.feature_importances_, index=numeric_cols).nlargest(10).plot(kind='barh', color='skyblue')
plt.title('Key Drivers of Country Segmentation')
plt.show()